In [66]:
import scipy
import pylab
import numpy as np
import matplotlib.pylab as plt

from scipy.fft import fft, ifft
from IPython.display import display
from typing import Optional, Tuple, List
from chipwhisperer.common.traces import Trace

In [45]:
%run '../utils/save_data.ipynb'

In [67]:
Record = Tuple[Trace, Optional[str], Optional[str]]

In [68]:
SimpleRecord = Tuple[List[float], Optional[str], Optional[str]]

In [71]:
def plot_scope_traces(traces_data: List[Record], settings: dict, time_axis=True) -> None:
    """Plots scope output traces for examination using matplotlib.
    Used in jupyter with the `%matplotlib ipympl` magic keywork.
    
    Args:
    	traces_data ([(wave, description, color)]): trace to be plotted
            - wave (DataObject): trace to be plot
            - description (str): trace label
            - color (optional[str]): trace color
    	time_axis (bool): If true, converts x-axis from sample domain to time domain using adc frequency conversion.
    
    Returns:
    	none: renders plots on output
    """
    x_domain = np.arange(settings['num_samples'])
    title = "Traces"
    x_axis_label = "Sample"
    y_axis_label = "ADC Output"

    # Converting sample domain to time domain
    if time_axis:
        x_domain = (x_domain / settings['adc_frequency_Hz']) * (1e6)
        x_axis_label = "Time (us)"
        
    render_traces(x_domain, traces_data, title=title, x_axis_label=x_axis_label, y_axis_label=y_axis_label)

In [72]:
def plot_traces(x_domain: List[float], traces_data: List[Record]) -> None:
    """Plots generated output traces for examination using matplotlib.
    Used in jupyter with the `%matplotlib ipympl` magic keywork.
    
    Args:
    	x_domain (list(any)): create 
    	traces_data (tuple(list(any), str, optional(str))): trace to be plotted
    
    Returns:
    	none: renders plots on output
    """
    render_traces(x_domain, traces_data)

In [74]:
def render_traces(x_domain: List[float], y_domain: List[Record], title="Traces", x_axis_label="Input", y_axis_label="Output") -> None:
    """Renders generated output traces for examination using matplotlib.
    Used in jupyter with the `%matplotlib ipympl` magic keywork.

    Args:
    	x_domain (list(int)): x-domain values
    	y_domain (list(any)): normal contains tuple of values to be plotted

    Returns:
    	none: renders plots on output
    """
    plt.clf()
    fig, ax = plt.subplots(figsize=(15,5))

    ax.set_xlabel(x_axis_label)
    ax.set_ylabel(y_axis_label)
    ax.set_title(title)
    ax.grid(True)

    legend_exists = True
    plotted_traces = []

    for trace, description, color in y_domain:
        legend_exists = legend_exists and description
        plotted_trace, = ax.plot(x_domain, trace.wave, label=description, color=color)
        plotted_traces.append(plotted_trace)

    if not legend_exists:
        plt.show()
        return
        
    leg = ax.legend(ncol=max(len(y_domain) / 15, 1), loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize='small', columnspacing=1)
    lined = {}

    for legline, origline in zip(leg.get_lines(), plotted_traces):
        legline.set_picker(True)  # Enable picking
        lined[legline] = origline

    def on_pick(event):
        legline = event.artist
        origline = lined[legline]
        visible = not origline.get_visible()
        origline.set_visible(visible)
        legline.set_alpha(1.0 if visible else 0.2)  # fade legend entry
        fig.canvas.draw()

    fig.canvas.mpl_connect("pick_event", on_pick)
    plt.show()

In [63]:
def render_plot(x_domain: List[float], y_domain: List[SimpleRecord], title="Traces", x_axis_label="Input", y_axis_label="Output", scatter_plot=False) -> None:
    """Renders generated output traces for examination using matplotlib.
    Used in jupyter with the `%matplotlib ipympl` magic keywork.

    Args:
    	x_domain (list(float)): x-domain values
    	y_domain (list(SimpleRecord))): list of tuples of values to be plotted

    Returns:
    	none: renders plots on output
    """
    plt.clf()
    fig, ax = plt.subplots(figsize=(15,5))

    ax.set_xlabel(x_axis_label)
    ax.set_ylabel(y_axis_label)
    ax.set_title(title)
    ax.grid(True)

    legend_exists = True
    plotted_traces = []

    for wave, description, color in y_domain:
        legend_exists = legend_exists and description
        if scatter_plot:
            plotted_trace = ax.scatter(x_domain, wave, label=description, color=color)
        else:
            plotted_trace, = ax.plot(x_domain, wave, label=description, color=color)
        plotted_traces.append(plotted_trace)

    if not legend_exists:
        plt.show()
        return
        
    leg = ax.legend(ncol=max(len(y_domain) / 15, 1), loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize='small', columnspacing=1)
    lined = {}

    for legline, origline in zip(leg.get_lines(), plotted_traces):
        legline.set_picker(True)  # Enable picking
        lined[legline] = origline

    def on_pick(event):
        legline = event.artist
        origline = lined[legline]
        visible = not origline.get_visible()
        origline.set_visible(visible)
        legline.set_alpha(1.0 if visible else 0.2)  # fade legend entry
        fig.canvas.draw()

    fig.canvas.mpl_connect("pick_event", on_pick)
    plt.show()

In [62]:
def render_plots(x_domain: List[float], y_domains: List[List[SimpleRecord]], title=["Traces"], x_axis_label="Input", y_axis_label="Output", scatter_plot=False) -> None:
    """Renders generated output traces for examination using matplotlib.
    Used in jupyter with the `%matplotlib ipympl` magic keywork.

    Args:
    	x_domain (list(float)): x-domain values
    	y_domains (list(list(SimpleRecord)))): list of tuples of values to be plotted

    Returns:
    	none: renders plots on output
    """
    plt.clf()
    #plt.figure(figsize=(15,5))
    #plt.figure()
    fig, axes = plt.subplots(nrows=len(y_domains), ncols=1, sharex=True, figsize=(15,5*len(y_domains)))
    index = 0
    
    for y_domain in y_domains:
        ax = axes[index]
    
        ax.set_xlabel(x_axis_label)
        ax.set_ylabel(y_axis_label)
        ax.set_title(title[index] if len(title) > index else title[0])
        ax.grid(True)
    
        legend_exists = True
        plotted_traces = []
    
        for wave, description, color in y_domain:
            legend_exists = legend_exists and description
            if scatter_plot:
                plotted_trace = ax.scatter(x_domain, wave)
            else:
                plotted_trace, = ax.plot(x_domain, wave, label=description, color=color)
            plotted_traces.append(plotted_trace)
    
        if not legend_exists:
            plt.show()
            return
            
        leg = ax.legend(ncol=max(len(y_domain) / 15, 1), loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize='small', columnspacing=1)
        lined = {}
    
        for legline, origline in zip(leg.get_lines(), plotted_traces):
            legline.set_picker(True)  # Enable picking
            lined[legline] = origline
    
        def on_pick(event):
            legline = event.artist
            origline = lined[legline]
            visible = not origline.get_visible()
            origline.set_visible(visible)
            legline.set_alpha(1.0 if visible else 0.2)  # fade legend entry
            fig.canvas.draw()
    
        fig.canvas.mpl_connect("pick_event", on_pick)
        index += 1
    plt.show()

In [43]:
def plot_scope_trace_FFT(settings, trace_data: List[Record]) -> None:
    """Renders the output of the FFT of the provided trace_data.

    Args:
        scope (object): chipwhisperer scope object
    	traces_data (tuple(list(any), str, optional(str))): trace to be plotted

    Returns:
        none: renders plot on output
    """
    plt.clf()

    (trace, description) = trace_data
    time_domain = (np.arange(settings['num_samples']) / settings['adc_frequency_Hz']) * (1e6)
    trace_FFT = abs(fft(trace.wave))
    freqs = scipy.fftpack.fftfreq(trace.wave.size, time_domain[1]-time_domain[0])
    
    plt.subplot(211)
    plt.plot(time_domain, trace.wave)
    plt.subplot(212)
    plt.plot(freqs,20*np.log10(trace_FFT),'x')
    plt.show()

In [44]:
def plot_reset() -> None:
    plt.close('all')